# Whole-Brain Meta-Analytic Map Clustering

This notebook investigates the global spatial relationships among the specific cognitive terms used to decode our VMPFC subregions. By generating whole-brain meta-analytic association maps (z-scores) for each term via Neurosynth, we can objectively measure how similarly these psychological concepts are represented across the entire human brain (excluding the VMPFC itself, to avoid circularity).

We apply hierarchical agglomerative clustering to these unROI spatial maps to test whether our predefined theoretical categories—Affect (Red), Valuation (Yellow), and Social Cognition (Blue)—naturally segregate into distinct, data-driven neurobiological networks.

In [11]:
import pandas as pd
from neurosynth import Dataset
from neurosynth.analysis.cluster import Clusterable
from copy import deepcopy
import joblib
from pathlib import Path
import nibabel as nib
import numpy as np
import nilearn.plotting as nplt
import matplotlib.pyplot as plt
from sklearn.metrics import adjusted_rand_score

import utils

import warnings
import json
import numpy as np
from scipy.cluster.hierarchy import fcluster, linkage
from sklearn.metrics import adjusted_rand_score
from collections import Counter
from neurosynth import meta

plt.rcParams['axes.grid'] = False
plt.rcParams['font.family'] = 'Arial'
warnings.filterwarnings("ignore", category=FutureWarning)

DATA_PATH = Path('../data')
RESULTS_PATH = Path('../results')
PLOTS_PATH = Path('../plots')
DOCUMENTS_PATH = Path('../documents')
DATASET_FILE = DATA_PATH / 'neurosynth_data/dataset.pkl'
unROI_FILE = DATA_PATH / 'masks/unVMPFC_mask_v2.nii'
PLOT_KWARGS_DICT = dict(dpi=300, transparent=True, bbox_inches='tight')

In [12]:
dataset = Dataset.load(DATASET_FILE)

# load mask image
unROI_IMAGE = nib.load(str(unROI_FILE))
unroi_mask = unROI_IMAGE.get_fdata() > 0

ROI mask includes 3710 voxels.
unROI mask includes 224756 voxels.


In [13]:
# get term and term mapping
category_color_mapping_dict = {'affect': '#c00000',  'valuation': '#eab200','social': '#0036d9', }
with open(DATA_PATH / 'neurosynth_data/term_mapping.json', 'r', encoding='utf8') as f:
    term_mapping_dict = json.load(f)
    term_list = list(term_mapping_dict.keys())
# Load Meta-Analytic Maps
term_brain_map_dict = {}
for term in term_list:
    ids = dataset.get_studies(features=[term], frequency_threshold=0.001)
    ma = meta.MetaAnalysis(dataset, ids)
    term_brain_map_dict[term] = ma.images['association-test_z']

In [14]:
term_mapping_dict_50 = {k: v for (k, v) in term_mapping_dict.items() if v != "others"}
term_list_50 = list(term_mapping_dict_50.keys())
print(f'Length of "term_list_50": {len(term_list_50)}, and terms are:')
print(', '.join(term_list_50), '\n')

term_list_15 = term_list_50[:15]
term_mapping_dict_15 = {k: v for k, v in term_mapping_dict.items() if k in term_list_15}
print(f'Length of "term_list_15": {len(term_list_15)}, and terms are:')
print(', '.join(term_list_15))

Length of "term_list_50": 40, and terms are:
autobiographical, reward, social, emotion, value, self, decision making, affective, person, fear, theory mind, food, choice, depression, motivation, autonomic, arousal, emotion regulation, reinforcement, empathy, threat, incentive, anxiety, autism, conditioning, social interactions, major depressive, happy, faces, monetary, pain, impulsive, drug, feelings, mood, loss, gambling, self referential, aversive, stress 

Length of "term_list_15": 15, and terms are:
autobiographical, reward, social, emotion, value, self, decision making, affective, person, fear, theory mind, food, choice, depression, motivation


# Hierarchical Clustering: Top 15 Terms

We first evaluate the spatial hierarchy of our core set of 15 highly relevant cognitive terms. We extract the unmasked whole-brain activation vector for each term, calculate their pairwise Pearson correlation distances, and apply average-linkage hierarchical clustering.

The resulting circular dendrogram visually illustrates the global functional architecture. If our theoretical domain mapping is neurobiologically valid, terms belonging to the same cognitive domain (sharing the same color) should naturally group together into distinct, contiguous branches.

In [ ]:
# 1. whole brain
brain_matrix_15 = np.array([dataset.masker.unmask(term_brain_map_dict[t])[unroi_mask] for t in term_mapping_dict_15])
Z_15 = linkage(brain_matrix_15, method='average', metric='correlation')
term_color_mapping_dict_15 = {k: category_color_mapping_dict[v] for (k, v) in term_mapping_dict_15.items()}
fig, ax = utils.plot.circular_dendrogram(
    Z_15, term_color_mapping_dict_15,
    figsize=(4, 4), line_width=2, font_size=14, gap_fraction=1/len(term_color_mapping_dict_15), r_inner=0.1,
    rotation_degrees=-55, flip=True
)
fig.savefig(PLOTS_PATH / 'Meta-Analysis_terms_15.png', **PLOT_KWARGS_DICT)
fig.savefig(PLOTS_PATH / 'Meta-Analysis_terms_15.svg', **PLOT_KWARGS_DICT)

# Hierarchical Clustering: Extended Term Set (Top 50)

To confirm that the functional segregation observed in the 15-term dendrogram is robust and not an artifact of selective sampling, we scale up our spatial clustering analysis to a broader dictionary of 50 cognitive terms.

Applying the exact same correlation and hierarchical clustering pipeline, we generate a comprehensive circular dendrogram. This larger topological tree provides robust evidence for how broad psychological constructs systematically diverge across the whole-brain coordinate space.

In [ ]:
# 1. whole brain
brain_matrix_50 = np.array([dataset.masker.unmask(term_brain_map_dict[t])[unroi_mask] for t in term_mapping_dict_50])
Z_50 = linkage(brain_matrix_50, method='average', metric='correlation')

term_color_mapping_dict_50 = {k: category_color_mapping_dict[v] for (k, v) in term_mapping_dict_50.items()}
fig, ax = utils.plot.circular_dendrogram(
    Z_50, term_color_mapping_dict_50,
    figsize=(6, 6), line_width=2, font_size=14, gap_fraction=1/len(term_color_mapping_dict_50), r_inner=0., r_outer=1, rotation_degrees=-70, n_clusters=4,
)

fig.savefig(PLOTS_PATH / 'Meta-Analysis_terms_50.png', **PLOT_KWARGS_DICT)
fig.savefig(PLOTS_PATH / 'Meta-Analysis_terms_50.svg', **PLOT_KWARGS_DICT)